- https://gemini.google.com/app/8b6ab0d1fd30c9ff
- https://chatgpt.com/c/695ca49a-1204-8321-a140-e53f3df7410c

- rlhf, rft, rlvr
    - RLHF 侧重于“对齐人类的主观偏好”（如语气、安全），而 RFT 和 RLVR 侧重于“提升客观的推理能力”（如数学、代码）。
    - rft: Reinforced Fine-Tuning
        - 拒绝采样微调 (Rejection Sampling Fine-Tuning)。机制：让模型对同一个问题生成 N 个回答，用一个打分器（可以是奖励模型，也可以是验证器）选出最好的一个或几个，然后把这些“好样本”当作标准答案，对模型进行监督微调（SFT）。这是一种“离线”方法。
- rft 最终虽然也是 sft，但很重要的一点是，rft 的数据，是 base model 自己 rollout 采样出来的，类 on-policy data

### align tax 与灾难性遗忘

- 机器学习的传统假设是数据独立同分布（i.i.d.）。在针对性的顺序后训练中，这一假设被打破。如果一个 LLM 先针对任务 A（通用对话）训练，再针对任务 B（数学）训练，最后针对任务 C（代码）进行 RL 训练，为任务 C 计算的梯度并不天然尊重任务 A 和 B 所需的流形（Manifold）。若无干预，任务 C 的优化轨迹极有可能将权重推向与任务 A 所需方向正交甚至相反的方向。这种现象即为灾难性遗忘。
- 在标准的 RLHF（特别是近端策略优化 PPO）中，目标函数包含一个基于 Kullback-Leibler (KL) 散度的惩罚项，用于约束当前策略 $\pi_\theta$ 与参考模型 $\pi_{ref}$（通常是 SFT 模型）之间的距离。
    - $L_{total} = E [ r(x, y) - \beta \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)} ]$
    - 理论上，这个 KL 惩罚项的设计初衷正是为了防止模型偏离基础能力太远，从而保留通用知识。然而，实际操作中存在两个严重问题：参考模型的局限性： 参考模型 $\pi_{ref}$ 通常已经是经过 SFT 的模型，它本身可能已经开始向特定任务偏移。奖励黑客（Reward Hacking）与分布坍缩： 随着 RL 步数增加以最大化特定任务（如数学）的奖励 $r(x, y)$，模型会利用奖励函数的漏洞，生成高回报但低概率的特定模式。这种优化会导致输出分布的熵值显著降低（Mode Collapse），即模型变得“死板”。研究表明，RLHF 后的模型在输出多样性上显著低于 SFT 模型。这种多样性的丧失意味着模型失去了在通用对话中所需的“灵活性”和“创造力”，这是对齐税的直接体现。

### rft

- references
    - https://arxiv.org/abs/2507.05386
        - Reinforcement Fine-Tuning Naturally Mitigates Forgetting in Continual Post-Training
    - https://arxiv.org/pdf/2509.12235
        - RL FINE-TUNING HEALS OOD FORGETTING IN SFT
- 顺序 SFT（先训练任务 A，再训练任务 B）具有高度的破坏性。
    - 机制分析： SFT 使用交叉熵损失（Cross-Entropy Loss），强制模型最大化训练数据中特定 Token 的概率。这提供了一个强硬的、无差别的梯度信号，迫使模型权重积极适应新分布 $P(y|x)_{new}$。
    - 后果： 这种激进的适应会覆盖编码先前任务的权重。实验表明，顺序 SFT 会导致先前学习任务的性能严重退化（遗忘度量 FM 高达 -10.4%）。模型本质上是用新数据“重写”了其硬盘。
- RFT（使用 PPO 或 GRPO 等算法）表现出一种“天然”的抗遗忘性。
    - 方差驱动的更新： RFT 的更新是由奖励信号和**优势函数（Advantage Function）**驱动的。梯度更新的幅度与奖励的方差成正比。
        - 如果模型生成的输出已经很好（高概率、高奖励），优势函数较小，更新幅度微乎其微。
        - 只有当模型不确定或生成质量较差时，更新才会显著。
    - 自我轨迹采样： 关键在于，RFT 采样的是模型自身的轨迹。它并不强迫模型完美模仿外部数据集（如 SFT 那样），而是鼓励模型找到一种能满足奖励的解。这种机制允许模型在只要能获得高奖励的前提下，尽可能保留其预训练的内部结构。
    - 数据依赖的正则化器： 奖励方差充当了一种数据依赖的正则化器。RFT 中的梯度更新有效地过滤掉了那些对奖励优势没有贡献的“不必要”权重偏移，从而保护了编码通用能力的“休眠”权重 。